In [2]:
import re
from pathlib import Path

import pandas as pd

df = pd.read_parquet(Path("..") / "data" / "processed" / "clean.parquet")
print(df.shape)
print(df.dtypes.astype(str).value_counts().to_string())


def is_text(s):
    return (
        pd.api.types.is_string_dtype(s)
        or pd.api.types.is_object_dtype(s)
        or isinstance(s.dtype, pd.CategoricalDtype)
    )


rows = []
for c in df.columns:
    s = df[c]
    if is_text(s):
        ss = s.dropna().astype(str)
        rows.append((
            c, str(s.dtype), int(s.nunique()),
            round(float(ss.str.len().mean()), 1) if len(ss) else 0.0,
            int(s.isna().sum()), ss.iloc[0][:50] if len(ss) else "",
        ))
text_cols = pd.DataFrame(rows, columns=["column", "dtype", "n_unique", "mean_len", "n_nan", "example"])
text_cols = text_cols.sort_values("mean_len", ascending=False)
print("\nТекстовые и категориальные столбцы:", len(text_cols))
print(text_cols.head(25).to_string(index=False))

bin_cols = [c for c in df.columns if not is_text(df[c]) and df[c].nunique(dropna=True) == 2]
print("\nБинарных столбцов:", len(bin_cols))
print([c for c in bin_cols if not re.search(r"_(19|20)\d{2}$", c)][:40])

year_bin = sorted({re.sub(r"_(19|20)\d{2}$", "", c) for c in bin_cols if re.search(r"_(19|20)\d{2}$", c)})
print("\nБинарные многолетние основы:", year_bin[:30])

date_like = [c for c in df.columns if re.search(r"year|yr|date|time", c, re.I) and not re.search(r"_(19|20)\d{2}$", c)]
print("\nСтолбцы про год или дату:", date_like[:25])

num_single = [
    c for c in df.columns
    if pd.api.types.is_numeric_dtype(df[c]) and not re.search(r"_(19|20)\d{2}$", c)
]
print("\nЧисловых столбцов без года в имени:", len(num_single))

(1089, 2619)
float32     1282
int64        675
float64      518
int32         97
int16         17
bool          13
category      11
str            6

Текстовые и категориальные столбцы: 17
         column    dtype  n_unique  mean_len  n_nan                                            example
ns_tpa_nam_2025      str       325      77.3    737 Dolphin Park ; Burdon Canal ; Krooman ; Yarborough
ns_tpa_des_2025      str       198      56.0    737 Public Reserve ; Nature Reserve ; Public Reserve ;
ns_mpa_nam_2025      str        64      48.3   1013 Seaflower ; Area Marina Protegida de la Reserva de
ns_mpa_des_2025      str        49      43.5   1013 Specially Protected Area (Cartagena Convention) ; 
ge_scl_lst_2023 category       482      40.5      2 Calcisols, Cambisols, Fluvisols, Leptosols, Luviso
wa_gwr_gwb_2025 category         3      28.0     17                        Local and Shallow Acquifers
ge_eco_cla_2025 category       138      22.7      0              California coastal sage a

In [3]:
import re
from collections import Counter

pat = re.compile(r"^(?P<stem>.+)_(?P<year>(?:19|20)\d{2})$")
years_by_stem = {}
for c in df.columns:
    m = pat.match(c)
    if m:
        years_by_stem.setdefault(m.group("stem"), []).append(int(m.group("year")))
snap = {s: y for s, y in years_by_stem.items() if len(y) == 1}
multi = {s: y for s, y in years_by_stem.items() if len(y) > 1}
no_year = [c for c in df.columns if not pat.match(c)]
print("основ с одним годом (снимки):", len(snap), "| многолетних:", len(multi), "| столбцов без года:", len(no_year))
print("годы снимков:", Counter(y[0] for y in snap.values()).most_common())

prefix = lambda c: c.split("_")[0]
num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c])]
print("\nчисловых столбцов по листам:", dict(Counter(prefix(c) for c in num)))

bools = [c for c in df.columns if pd.api.types.is_bool_dtype(df[c])]
print("\nbool:", bools)
bin_int = [c for c in num if df[c].nunique(dropna=True) == 2]
print("\nбинарные числовые:", len(bin_int), bin_int[:40])
print("\nстолбцы без года:", no_year[:40])

основ с одним годом (снимки): 183 | многолетних: 296 | столбцов без года: 15
годы снимков: [(2025, 81), (2020, 73), (2019, 12), (2024, 6), (2023, 3), (1990, 3), (2030, 3), (2005, 1), (2021, 1)]

числовых столбцов по листам: {'id': 1, 'gc': 5, 'cl': 102, 'em': 583, 'ex': 984, 'ge': 18, 'gh': 473, 'xst': 21, 'gr': 56, 'hz': 76, 'hl': 10, 'in': 10, 'lu': 30, 'ns': 11, 'sd': 36, 'sc': 154, 'wa': 19}

bool: ['dup_key_flag', 'sd_lue_lpr_1975_1980_cens', 'sd_lue_lpr_1980_1990_cens', 'sd_lue_lpr_1990_2000_cens', 'sd_lue_lpr_2000_2010_cens', 'sd_lue_lpr_2010_2020_cens', 'sd_lue_lpr_2020_2030_cens', 'sd_lue_lpr_1975_2020_cens', 'sd_lue_lpr_1990_2020_cens', 'sd_lue_lpr_2000_2020_cens', 'out_of_range', 'gc_pop_tot_2025_out', 'gc_uca_km2_2025_out']

бинарные числовые: 28 ['gc_ucb_yod_2025', 'gc_ucm_cap', 'ge_bio_a01_2025', 'ge_bio_a04_2025', 'ge_bio_a05_2025', 'ge_bio_a14_2025', 'ge_bio_a15_2025', 'gh_xst_a22_2005', 'gh_xst_a22_2010', 'gh_xst_s22_2005', 'gh_xst_s22_2010', 'gh_xst_v22_2005', 'gh_xst